# TradeSense — Layer 2 Benchmarks: Final Source + Read-Only Verification

The completed **`processed_layer2_final_v3`** folder is preserved. First audit existing artifacts (read-only). The final corrected source code is included for reproducibility but **does not run automatically**; to rebuild, use a **new versioned output directory** and explicitly invoke `main()`. Mt.Gox sequential row-index splits are **not** verified chronological or entity-independent splits. Sapienza's 1-hour embargo is heuristic.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Colab unavailable: set BASE to your local mounted dataset path.')
BASE = Path('/content/drive/MyDrive/research datasets')
assert BASE.exists(), f'Dataset root not mounted: {BASE}'


In [ ]:
ROOT = BASE / 'processed_layer2_final_v3'
assert ROOT.is_dir(), f'Layer 2 final folder not found: {ROOT}'
files = sorted(ROOT.glob('*'))
for p in files: print(p.name, f'{p.stat().st_size/1024/1024:.2f} MB')
for dataset, target in [('sapienza','gt'),('mtgox','wash')]:
    for split in ('train','val','test'):
        p = ROOT / f'{dataset}_{split}.parquet'
        assert p.exists(), f'Missing artifact: {p}'
        df = pd.read_parquet(p)
        assert target in df.columns, f'{p.name}: missing target'
        predictors = df.drop(columns=[target]).select_dtypes(include='number')
        assert not predictors.empty
        assert np.isfinite(predictors.to_numpy(dtype=float)).all(), f'{p.name}: non-finite predictors'
        assert df[target].isin([0,1]).all(), f'{p.name}: invalid labels'
        print(f'PASS {p.name}: {len(df):,} rows, {len(predictors.columns)} numeric predictors, label counts={df[target].value_counts().to_dict()}')
print('Layer 2 artifacts inspected. This does not independently verify original raw-data label quality.')


## Original final corrected pipeline source (manual rebuild only)

This code is preserved from your uploaded final Layer 2 cell. It is **not run automatically** and refuses to overwrite the existing output. If you ever need a rebuild, edit `OUTPUT` to a new versioned path, check input paths, and call `main()` manually.

In [ ]:
"""TradeSense Component 3 / Layer 2: corrected benchmark partitioning.

Sapienza: chronological, pump_index-group-disjoint split, 1-hour embargo.
Mt.Gox: index-order exploratory split ONLY; chronology/entity independence unverified.
Outputs are written to a NEW folder and never overwrite existing files.
Run in Google Colab after mounting Google Drive.
"""
from pathlib import Path
import json
import shutil
import uuid
import numpy as np
import pandas as pd
import joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

BASE = Path('/content/drive/MyDrive/research datasets')
SAP_FILE = BASE / 'pump-and-dump-dataset-master/labeled_features/features_5S.csv/features_5S.csv'
GOX_FILE = BASE / 'gox_ml_samples.csv'  # Edit if your file lives elsewhere
OUTPUT = BASE / 'processed_layer2_final_v3'
EMBARGO = pd.Timedelta(hours=1)  # Assumption, not a verified lookback bound
RUN_SAPIENZA = True
RUN_MTGOX = True

SAP_BASE = ['std_rush_order', 'avg_rush_order', 'std_trades', 'std_volume',
            'avg_volume', 'std_price', 'avg_price', 'avg_price_max',
            'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos']
GOX_BASE = ['bitcoins', 'price_deviation', 'buyer_24h_trade_count', 'seller_24h_trade_count']


def features(df, dataset):
    cols = SAP_BASE if dataset == 'sapienza' else GOX_BASE
    missing = set(cols) - set(df.columns)
    if missing:
        raise ValueError(f'{dataset}: missing predictor columns: {missing}')
    x = df[cols].apply(pd.to_numeric, errors='coerce').copy()
    if dataset == 'sapienza':
        x['rush_volume_interaction'] = x['avg_rush_order'] * x['avg_volume']
        x['log_avg_volume'] = np.log1p(x['avg_volume'].clip(lower=0))
        x['log_std_price'] = np.log1p(x['std_price'].clip(lower=0))
        assert len(x.columns) == 15
    else:
        x['trade_count_ratio'] = x['buyer_24h_trade_count'] / (x['seller_24h_trade_count'] + 1)
        x['log_bitcoins'] = np.log1p(x['bitcoins'].clip(lower=0))
        x['abs_price_deviation'] = x['price_deviation'].abs()
        assert len(x.columns) == 7
    return x.replace([np.inf, -np.inf], np.nan)


def report_parts(parts, target, name):
    print(f'\n===== {name.upper()} FINAL SPLIT =====')
    for part, frame in parts.items():
        counts = frame[target].value_counts(dropna=False).to_dict()
        print(f'{part:5s}: {len(frame):>10,} rows; labels={counts}')
        if len(frame) == 0 or not frame[target].isin([0, 1]).all() or frame[target].nunique() != 2:
            raise ValueError(f'{name}/{part}: both valid classes are required')


def sapienza_split():
    df = pd.read_csv(SAP_FILE)
    required = {'date', 'pump_index', 'gt'} | set(SAP_BASE)
    if required - set(df.columns):
        raise ValueError(f'Sapienza missing columns: {required - set(df.columns)}')
    df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')
    df['pump_index'] = pd.to_numeric(df['pump_index'], errors='coerce')
    if df['date'].isna().any() or not df['gt'].isin([0, 1]).all():
        raise ValueError('Sapienza invalid date or target')

    # FIX: Replaced df.gt.eq(1) with df['gt'] == 1
    if (df.loc[df['gt'] == 1, 'pump_index'].isna() |
        df.loc[df['gt'] == 1, 'pump_index'].lt(0)).any():
        raise ValueError('Some positive observations have no nonnegative pump_index')

    df = df.sort_values('date', kind='mergesort').reset_index(drop=True)
    event = df['pump_index'].ge(0)
    groups = (df.loc[event].groupby('pump_index')['date']
              .agg(start='min', end='max', n='size')
              .sort_values(['start', 'end']))
    if len(groups) < 20:
        raise ValueError('Too few event groups for splitting')

    cut1 = groups['start'].iloc[int(len(groups) * .70)]
    cut2 = groups['start'].iloc[int(len(groups) * .85)]
    if cut1 >= cut2 or cut2 - cut1 <= 2 * EMBARGO:
        raise ValueError('Invalid/too-close temporal cutoffs')
    lo1, hi1 = cut1 - EMBARGO, cut1 + EMBARGO
    lo2, hi2 = cut2 - EMBARGO, cut2 + EMBARGO

    train_ids = set(groups.index[groups['end'] < lo1])
    val_ids = set(groups.index[(groups['start'] >= hi1) & (groups['end'] < lo2)])
    test_ids = set(groups.index[groups['start'] >= hi2])
    assert not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids)

    background = ~event
    parts = {
        'train': df.loc[(event & df.pump_index.isin(train_ids)) |
                        (background & df.date.lt(lo1))].copy(),
        'val': df.loc[(event & df.pump_index.isin(val_ids)) |
                      (background & df.date.ge(hi1) & df.date.lt(lo2))].copy(),
        'test': df.loc[(event & df.pump_index.isin(test_ids)) |
                       (background & df.date.ge(hi2))].copy(),
    }
    report_parts(parts, 'gt', 'sapienza')
    for label, part in parts.items():
        if not part.date.is_monotonic_increasing:
            raise AssertionError(f'{label} is not time-sorted')
        # FIX: Replaced part.gt.eq(1) with part['gt'] == 1
        print(label, 'positive pump_index groups:', part.loc[part['gt'] == 1, 'pump_index'].nunique(),
              'date range:', part.date.min(), '->', part.date.max())
    if not (parts['train'].date.max() < parts['val'].date.min() < parts['test'].date.min()):
        raise AssertionError('Temporal partition ranges overlap')
    kept = sum(len(p) for p in parts.values())
    print('Cutoffs:', cut1, cut2, '| embargo:', EMBARGO)
    print('Rows excluded (embargo + boundary-spanning event groups):', len(df) - kept)
    print('Event groups excluded:', len(groups) - len(train_ids | val_ids | test_ids))
    return parts, [
        'Sapienza is sorted by parsed date; naive timestamps interpreted as UTC (source timezone unverified).',
        'pump_index>=0 treated as an event-group ID; original documentation semantics not independently verified.',
        'Each retained pump_index group belongs to only one partition; spanning groups are excluded entirely.',
        'One-hour embargo is heuristic, not proven to eliminate all feature-history dependence.',
        'Positive rows are not interchangeable with independent event counts.',
    ]


def mtgox_split():
    df = pd.read_csv(GOX_FILE, usecols=lambda c: c in GOX_BASE + ['wash'])
    if set(GOX_BASE + ['wash']) - set(df.columns):
        raise ValueError('Mt.Gox source is missing required columns')
    n = len(df)
    a, b = int(n * .70), int(n * .85)
    parts = {'train': df.iloc[:a].copy(), 'val': df.iloc[a:b].copy(), 'test': df.iloc[b:].copy()}
    report_parts(parts, 'wash', 'mtgox')
    return parts, [
        'Sequential SOURCE ROW INDEX split only; NOT a verified chronological or group-independent split.',
        'Source contains no timestamp, transaction ID, or entity ID to verify entity leakage.',
        'Historical 24h aggregates may share underlying observations across partitions.',
        'Use results only as exploratory within-dataset benchmarks, not leakage-free prospective performance.',
    ]


def export_dataset(parts, dataset, target, limitations, stage):
    x = {k: features(v, dataset) for k, v in parts.items()}
    columns = x['train'].columns.tolist()
    if any(v.columns.tolist() != columns for v in x.values()):
        raise ValueError('Feature schema differs across splits')
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    scaler = RobustScaler()
    train_imputed = imputer.fit_transform(x['train'])
    scaler.fit(train_imputed)
    y = parts['train'][target].astype(int)
    weight = float((y == 0).sum() / (y == 1).sum())
    for split, frame in parts.items():
        arr = scaler.transform(imputer.transform(x[split]))
        if not np.isfinite(arr).all():
            raise ValueError(f'{dataset}/{split}: non-finite predictors')
        out = pd.DataFrame(arr, columns=columns, dtype='float32')
        out[target] = frame[target].astype('int8').to_numpy()
        out.to_parquet(stage / f'{dataset}_{split}.parquet', index=False)
        print(f'Saved {dataset}_{split}.parquet:', out.shape)
        del arr, out
    joblib.dump({'features': columns, 'imputer': imputer, 'scaler': scaler,
                 'scale_pos_weight': weight, 'limitations': limitations},
                stage / f'{dataset}_metadata.joblib')
    print(dataset, 'training scale_pos_weight:', round(weight, 4))


def main():
    if OUTPUT.exists():
        raise FileExistsError(f'Refusing to overwrite existing output: {OUTPUT}')
    if RUN_SAPIENZA and not SAP_FILE.is_file():
        raise FileNotFoundError(SAP_FILE)
    if RUN_MTGOX and not GOX_FILE.is_file():
        raise FileNotFoundError(GOX_FILE)
    stage = BASE / ('.layer2_stage_' + uuid.uuid4().hex)
    stage.mkdir(parents=True, exist_ok=False)
    try:
        if RUN_SAPIENZA:
            parts, notes = sapienza_split()
            export_dataset(parts, 'sapienza', 'gt', notes, stage)
            del parts
        if RUN_MTGOX:
            parts, notes = mtgox_split()
            export_dataset(parts, 'mtgox', 'wash', notes, stage)
            del parts
        stage.rename(OUTPUT)
        print('\nSUCCESS: Final exports saved to:', OUTPUT)
        print('WARNING: Mt.Gox temporal/entity leakage remains UNVERIFIABLE with this source.')
    except Exception:
        shutil.rmtree(stage, ignore_errors=True)
        raise


# Deliberately NOT executed automatically. Existing v3 outputs must not be overwritten.
# To rebuild from raw sources into a NEW versioned OUTPUT folder, edit OUTPUT above,
# verify inputs and explicitly run: main()